In [9]:
import os
import sys
sys.path.append(os.path.expanduser('~/albatros_analysis'))
import numpy as np
import cupy as cp
import helper as hp_f
import figures as fgs
import json
import argparse
from src.utils import orbcomm_utils as outils
from src.utils import orbcomm_utils_gpu as outils_g
from src.utils import baseband_utils as butils
from src.correlations import baseband_data_classes as bdc
from scripts.orbcomm import sat_utils as su
from scripts.orbcomm import sat_utils_gpu as sug
from scripts.xcorr import helper as hp_x
import matplotlib.pyplot as plt

In [10]:
config_file = '/home/thomasb/albatros_analysis/scripts/orbcomm/config3_corr.json'
T_SPECTRA = 4096/250e6

v_acclen = 1000  #accumulation length for visibility sanity-checks
bline_ants = ['Antenna 1', 'Antenna 6']
file_save_names = ['MARS1', 'MARS6']
sat = 57166
chan_big_idx = 1837

pulse_rel_start_t = 16545
pulse_rel_end_t = 16810
buffer_start = 0
buffer_end = 0

In [11]:
dir_parents, coords, ant_names, clock_offsets = [], [], [], []

with open(config_file, "r") as f:
    config = json.load(f)
    for i, (ant, details) in enumerate(config["antennas"].items()):
        coords.append(details['coordinates'])
        dir_parents.append(details["path"])
        ant_names.append(details["name"])
        clock_offsets.append(details['clock_offset'])
    global_start_t = config["correlation"]["start_timestamp"]
    global_end_t = config["correlation"]["end_timestamp"]

In [12]:
t1 = pulse_rel_start_t + global_start_t + buffer_start
t2 = pulse_rel_end_t + global_start_t - buffer_end

tle_path = outils.get_tle_file(t1, "/project/rrg-sievers/mohanagr/OCOMM_TLES")

ant1_idx, ant2_idx = ant_names.index(bline_ants[0]), ant_names.index(bline_ants[1])
ant1_path, ant2_path = dir_parents[ant1_idx], dir_parents[ant2_idx]
ant1_coords, ant2_coords  = coords[ant1_idx], coords[ant2_idx]
ant1_offset, ant2_offset = clock_offsets[ant1_idx], clock_offsets[ant2_idx]
spec_offset = ant2_offset - ant1_offset

print('ant indices:', ant1_idx, ant2_idx)
print('ant paths:', ant1_path, ant2_path)
print('ant coords:', ant1_coords, ant2_coords)
print('clock offsets (wrt MARS1):', ant1_offset, ant2_offset)
print('relative clock offset (ant2 - ant1):', spec_offset)

ant indices: 0 4
ant paths: /scratch/mohanagr/summer_2025/baseband/mars1 /scratch/mohanagr/summer_2025/baseband/mars6
ant coords: [79.41717895, -90.76721818, 188.095] [79.3979952, -90.799868, 42.692]
clock offsets (wrt MARS1): 0 -901873
relative clock offset (ant2 - ant1): -901873


In [ ]:
blk_nspec = 10**6 
spectrum_indices = np.arange(blk_nspec)
T_BLOCK = blk_nspec * T_SPECTRA
nblks = int(np.floor((t2-t1)/T_BLOCK))

print('block period', T_BLOCK)
print('number of blocks in pulse', nblks)

ant1_files, ant1_file_idx, ant2_files, ant2_file_idx = hp_x.get_init_info_2ant(t1, 
                                                                               t2, 
                                                                               spec_offset, 
                                                                               ant1_path, 
                                                                               ant2_path)

#SET UP CHANNELS
channels = np.asarray(bdc.get_header(ant1_files[0])["channels"],dtype='int64')
chanstart = np.where(channels == 1834)[0][0]
chanend = np.where(channels == 1852)[0][0]
nchans = chanend - chanstart
chanlist = np.arange(1834, 1852)
chan_s_idx = np.where(chanlist == chan_big_idx)[0]
print('chanstart, chanend:', chanstart, chanend)
print('small channel index', chan_s_idx)

block period 16.384
number of blocks in pulse 16
took 0.112 seconds to read raw data on  /scratch/mohanagr/summer_2025/baseband/mars1/17532/1753216692.raw
took 0.109 seconds to read raw data on  /scratch/mohanagr/summer_2025/baseband/mars6/17532/1753216676.raw
before correction 183105 1159668
index 1: 1120746
after correction 183105 1120746
Not reading any data
chanstart, chanend: 284 302
small channel index [3]
ACCLEN RECEIVED IS 1000000
took 0.111 seconds to read raw data on  /scratch/mohanagr/summer_2025/baseband/mars1/17532/1753216692.raw
START SPECNUM IS 1010926400 obj start at 1010926400
ACCLEN RECEIVED IS 1000000
took 0.108 seconds to read raw data on  /scratch/mohanagr/summer_2025/baseband/mars6/17532/1753216676.raw
START SPECNUM IS 1010890636 obj start at 1010890632


In [43]:
ant1_blks = []
ant2_blks = []

ant1 = bdc.BasebandFileIterator(ant1_files,
                                0,
                                ant1_idx,
                                blk_nspec,
                                nchunks=nblks,
                                chanstart=chanstart,
                                chanend=chanend,
                                type = 'float')

ant2 = bdc.BasebandFileIterator(ant2_files,
                                0,
                                ant2_idx,
                                blk_nspec,
                                nchunks=nblks,
                                chanstart=chanstart,
                                chanend=chanend,
                                type='float')

for i, (chunk1,chunk2) in enumerate(zip(ant1,ant2)):
    ant1_blk = chunk1['pol0'][:, chan_s_idx].ravel()
    ant2_blk = chunk2['pol0'][:, chan_s_idx].ravel()
    ant1_blks.append(ant1_blk)
    ant2_blks.append(ant2_blk)

ACCLEN RECEIVED IS 1000000
took 0.111 seconds to read raw data on  /scratch/mohanagr/summer_2025/baseband/mars1/17532/1753216692.raw
START SPECNUM IS 1010926400 obj start at 1010926400
ACCLEN RECEIVED IS 1000000
took 0.109 seconds to read raw data on  /scratch/mohanagr/summer_2025/baseband/mars6/17532/1753216676.raw
START SPECNUM IS 1010890636 obj start at 1010890632
print shape of my pols 1000000 18
print shape of my pols 1000000 18
print shape of my pols 1000000 18
print shape of my pols 1000000 18
print shape of my pols 1000000 18
print shape of my pols 1000000 18
print shape of my pols 278688 18
took 0.113 seconds to read raw data on  /scratch/mohanagr/summer_2025/baseband/mars1/17532/1753216744.raw
print shape of my pols 721312 18
print shape of my pols 278684 18
took 0.114 seconds to read raw data on  /scratch/mohanagr/summer_2025/baseband/mars6/17532/1753216730.raw
print shape of my pols 721316 18
print shape of my pols 1000000 18
print shape of my pols 1000000 18
print shape of